# Week 6: Spark Architecture & Data Processing Assignment

## Objective: 
Understand **Spark** architecture and perform efficient data processing using transformations, filtering, schema handling, and optimized file formats.

In [ ]:
import os
from pyspark.sql import SparkSession

# Initialize Spark
spark = SparkSession.builder.appName("GenerateAssignmentData").getOrCreate()

# Create dummy folder path
os.makedirs("data", exist_ok=True)

# Generate a list of fake order records matching assignment constraints
mock_data = [
    (101, "Prod_A", "Electronics", 1200.0, "Completed", 1200.0),
    (102, "Prod_B", "Furniture", 850.0, "Completed", 850.0),
    (103, "Prod_C", "Electronics", 1500.0, "Pending", 1500.0),
    (104, "Prod_D", "Electronics", 950.0, "Completed", 950.0),
    (None, "Prod_E", "Electronics", 2000.0, "Completed", 2000.0) # Null user entry
]

columns = ["user_id", "old_name", "category", "price", "status", "amount"]

# Create DataFrame and save as local test CSV
df_mock = spark.createDataFrame(mock_data, schema=columns)
df_mock.write.mode("overwrite").option("header", "true").csv("data/source.csv")

print("Setup Complete: Mock dataset generated inside 'data/source.csv' folder!")


In [ ]:
from pyspark.sql.types import DoubleType

print("--- Running Q3: Loading CSV Data ---")
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("data/source.csv")

print("--- Running Q6: Column Modifications (Rename & Cast) ---")
df_revised = df.withColumnRenamed("old_name", "new_name") \
               .withColumn("price", df["price"].cast(DoubleType()))

print("--- Running Q5, Q8, Q14: Multi-Condition Filtering ---")
# Filters rows based on category, completed status, and transactional limits
df_filtered = df_revised.filter(
    (df_revised.category == "Electronics") & 
    (df_revised.status == "Completed") & 
    (df_revised.amount > 1000)
)

print("--- Running Q10: Adding Tax Calculation Column ---")
df_final = df_filtered.withColumn("final_price", df_filtered.price * 1.18)

print("--- Running Q12: Handling Nulls and Preparing Pipeline Output ---")
# Drop records missing critical user identifiers
df_pipeline_ready = df_final.filter(df_final.user_id.isNotNull())

print("--- Running Q15: Safe Data Preview Output ---")
# Safely renders results into notebook interface
df_pipeline_ready.show(5)

#### Q1: Explain the roles of the Driver, Cluster Manager, and Executor in a Spark application. 

Ans:

**Driver**: The master node process. It runs the main application code, creates the SparkSession, builds the execution plan (DAG), and schedules tasks.

**Cluster Manager**: An external resource manager (YARN, Mesos, Standalone, or Kubernetes) that allocates physical cluster resources (CPU, RAM) to the Spark app.

**Executor**: Worker processes running on cluster nodes. They execute individual tasks assigned by the driver, process data, and store results in memory/disk.

#### Q2: How does Spark’s Lazy Evaluation strategy improve performance when chain-processing large datasets? 


Ans: 

Spark saves instructions as a logical plan (DAG) instead of executing them instantly. 
It only runs the data pipeline when an action is triggered, allowing it to optimize the steps and skip reading unnecessary data.

#### Q3: Write a Spark command to read a CSV file located at "data/source.csv", ensuring the first row is treated as a header and inferSchema is enabled. 

In [ ]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("data/source.csv")

#### Q4: What is the difference between CSV and Parquet in terms of storage (row-based vs. columnar) and why does it matter for performance?

Ans:

**CSV**: Row-based. Spark must scan every single column in a row even if your query only requests one.

**Parquet**: Columnar-based. It allows Spark to fetch only the specific columns needed, reducing disk reading and memory use.

### **Performance Insight**: Storage Formats
* **Disk & Network I/O:** Parquet uses columnar storage and heavy binary compression (Snappy/Gzip), which drastically shrinks file sizes compared to plain-text CSV.
* **Column Pruning:** If a query only reads 2 columns out of a 100-column dataset, Parquet fetches *only* those 2 columns from disk. CSV forces Spark to scan all 100 columns row-by-row, wasting massive amounts of memory and CPU cycle time.


#### Q5: Given a DataFrame df, write a query to select the columns product_id and price where the category is 'Electronics'. 

In [ ]:
df.select("product_id", "price").filter(df.category == "Electronics")

#### Q6: Write the code to "revise" a DataFrame by renaming the column old_name to new_name and casting the price column from a String to a Double. 

In [ ]:
from pyspark.sql.types import DoubleType

df_revised = df.withColumnRenamed("old_name", "new_name") \
               .withColumn("price", df["price"].cast(DoubleType()))

#### Q7: How does Spark use the Lineage Graph (DAG) to provide fault tolerance if a worker node fails?

Ans:

Spark tracks every operation applied to a dataset in a Lineage Graph (DAG). If a worker node crashes, Spark refers to this history and replays the transformations on a healthy node to rebuild only the lost data.

#### Q8: Write a query to filter a DataFrame df_orders for rows where the status is 'Completed' AND the amount is greater than 1000. 

In [ ]:
df_orders.filter((df_orders.status == "Completed") & (df_orders.amount > 1000))

#### Q9: Explain the concept of Predicate Pushdown in Parquet and how it affects the amount of data loaded into memory. 

Ans:

Parquet evaluates filtering conditions at the storage layer before loading rows into memory. It drops irrelevant blocks directly on disk so Spark only reads the exact data requested.

### **Performance Insight**: Predicate Pushdown
* **Storage-Level Filtering:** Instead of loading billions of rows into cluster memory and filtering them inside Spark, Predicate Pushdown passes the `filter()` logic straight to the Parquet file layer.
* **Skipping Blocks:** Parquet metadata tracks the minimum and maximum values of data chunks. If your filter is `age > 30` and a chunk only contains ages 10–20, Spark skips reading that entire file chunk from disk, cutting down I/O overhead to near zero.


#### Q10: Write a code snippet to add a new column final_price which is the base_price multiplied by 1.18 (18% tax).

In [ ]:
df_final = df.withColumn("final_price", df.base_price * 1.18)

#### Q11: What is the difference between Transformations and Actions? Provide two examples of each. 

Ans:

**Transformations**: Lazy operations that create a new DataFrame from an existing one without immediate evaluation (e.g., .filter(), .select()).
**Actions**: Operations that execute the computed DAG pipeline to yield or save a result (e.g., .count(), .show()).

#### Q12: Write the Spark command to load a Parquet file from "path/to/input", filter out any rows where user_id is null, and save the result as a CSV at "path/to/output".

In [ ]:
df = spark.read.parquet("path/to/input")
df_filtered = df.filter(df.user_id.isNotNull())
df_filtered.write.format("csv").option("header", "true").save("path/to/output")

#### Q13: In Spark Architecture, what is the difference between Client Mode and Cluster Mode? 

Ans: 

**Client Mode**: The driver program runs directly on your local machine, making it ideal for interactive testing and debugging.

**Cluster Mode**: The driver process runs inside a worker node on the cluster, preventing jobs from crashing if your local connection drops.

#### Q14: Write a query to filter a dataset for rows where the region is 'North' OR the priority is 'High'. 

In [ ]:
df.filter((df.region == "North") | (df.priority == "High"))

#### Q15: When exploring a dataset, why is it safer to use .show(5) instead of .collect() on a multi-terabyte dataset?

Ans:

**.show(5)** brings only 5 rows to the console for a quick, safe data preview.

**.collect()** forces Spark to dump the entire multi-terabyte dataset into the driver node’s memory at once, instantly crashing the app with an Out Of Memory (OOM) error.

### **Performance & Architecture Insight**: Memory Management
* **Driver Safety:** `.show(5)` sends a highly constrained execution instruction down the DAG to pull only the first 5 records over the network. 
* **OOM Prevention:** `.collect()` tells every single cluster executor node to send *all* data partitions to the single Driver node simultaneously. If your dataset size is larger than the Driver's allocated RAM (e.g., pulling a 50GB dataset into a 4GB driver), the application instantly crashes with a fatal `java.lang.OutOfMemoryError (OOM)`.
